In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

In [4]:
import os
import glob
 
tissue_3d = pd.read_csv("biomarkers_tissue_volumes.csv")
biomarkers_2d = pd.read_csv("biomarkers_2D.csv")
 
clinical_path = glob.glob("*longitudinal*.xlsx") + glob.glob(
    os.path.join("..", "src", "Data", "*longitudinal*.xlsx"))
clinical = pd.read_excel(clinical_path[0])
 
first_visit = clinical[clinical["Visit"] == 1].copy()
first_visit = first_visit[first_visit["Group"].isin(["Nondemented", "Converted"])]
first_visit = first_visit.rename(columns={"MRI ID": "subject_id"})
first_visit["label"] = first_visit["Group"].map({"Nondemented": 0, "Converted": 1})
 
data = first_visit[["subject_id", "label"]].merge(tissue_3d, on="subject_id", how="inner")
data = data.merge(biomarkers_2d, on="subject_id", how="inner")
print(f"{len(data)} sujets avec label + biomarqueurs 3D + 2D disponibles.")
print(data["label"].value_counts())

86 sujets avec label + biomarqueurs 3D + 2D disponibles.
label
0    72
1    14
Name: count, dtype: int64


In [6]:
feature_sets = {
    "Volumes 3D uniquement": ["gray_matter_mL", "white_matter_mL", "csf_mL", "total_brain_mL"],
    "Aires 2D uniquement": ["gray_matter_area_cm2", "white_matter_area_cm2", "csf_area_cm2"],
    "Asymétrie uniquement": ["asymmetry_gray_matter", "asymmetry_white_matter"],
    "Radiomique uniquement": ["intensity_mean", "intensity_std", "intensity_skew",
                               "intensity_kurtosis", "texture_contrast",
                               "texture_homogeneity", "texture_energy"],
    "Tout combiné": ["gray_matter_mL", "white_matter_mL", "csf_mL", "total_brain_mL",
                      "asymmetry_gray_matter", "asymmetry_white_matter",
                      "intensity_mean", "intensity_std", "texture_contrast"],
}
 
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = {}
 
for name, features in feature_sets.items():
    X = data[features].fillna(data[features].mean())
    y = data["label"]
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
    scores = cross_val_score(model, X, y, cv=skf, scoring="roc_auc")
    results[name] = scores
    print(f"{name:30s} AUC moyen = {scores.mean():.3f} (+/- {scores.std():.3f})")

Volumes 3D uniquement          AUC moyen = 0.437 (+/- 0.203)
Aires 2D uniquement            AUC moyen = 0.342 (+/- 0.181)
Asymétrie uniquement           AUC moyen = 0.261 (+/- 0.115)
Radiomique uniquement          AUC moyen = 0.463 (+/- 0.151)
Tout combiné                   AUC moyen = 0.410 (+/- 0.140)


In [7]:
print("\nCoefficients de la régression logistique (jeu combiné, sur tout l'échantillon) :")
X_all = data[feature_sets["Tout combiné"]].fillna(data[feature_sets["Tout combiné"]].mean())
y_all = data["label"]
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
model.fit(X_all, y_all)
coefs = pd.Series(model.named_steps["logisticregression"].coef_[0], index=feature_sets["Tout combiné"])
print(coefs.sort_values(key=abs, ascending=False))


Coefficients de la régression logistique (jeu combiné, sur tout l'échantillon) :
intensity_mean           -0.633812
asymmetry_gray_matter     0.345184
gray_matter_mL           -0.266569
texture_contrast         -0.246090
asymmetry_white_matter   -0.175183
total_brain_mL           -0.150078
white_matter_mL           0.038305
intensity_std             0.036156
csf_mL                    0.027981
dtype: float64


In [8]:
y_pred = cross_val_predict(model, X_all, y_all, cv=skf)
print("\nMatrice de confusion (jeu combiné, prédictions hors-échantillon) :")
print(confusion_matrix(y_all, y_pred))
print("\nRapport de classification :")
print(classification_report(y_all, y_pred, target_names=["Nondemented", "Converted"]))



Matrice de confusion (jeu combiné, prédictions hors-échantillon) :
[[43 29]
 [10  4]]

Rapport de classification :
              precision    recall  f1-score   support

 Nondemented       0.81      0.60      0.69        72
   Converted       0.12      0.29      0.17        14

    accuracy                           0.55        86
   macro avg       0.47      0.44      0.43        86
weighted avg       0.70      0.55      0.60        86

